In [1]:

import numpy as np
import time

from lot3_algorithmes import (
    acp_directe_fit,
    acp_iterative_fit
)


def generer_donnees_synthetiques(
    n=500,
    d=64,
    type_spectre="rapide",
    seed=42
):
    """
    Genere une matrice X centree avec un spectre controle.

    La covariance empirique de X possede exactement
    les valeurs propres choisies.
    """

    rng = np.random.default_rng(seed)

    # 1. Definir les valeurs propres
    if type_spectre == "rapide":
        eigenvalues = np.exp(-0.15 * np.arange(d))

    elif type_spectre == "lent":
        eigenvalues = np.exp(-0.02 * np.arange(d))

    else:
        raise ValueError("Type de spectre inconnu")

    # 2. Generer une base orthogonale dans R^d
    W, _ = np.linalg.qr(rng.normal(size=(d, d)))

    # 3. Generer une base orthonormale dans R^n
    A = rng.normal(size=(n, d))
    U, _ = np.linalg.qr(A)

    # 4. Construire X
    X = np.sqrt(n - 1) * U @ np.diag(
        np.sqrt(eigenvalues)
    ) @ W.T

    return X, eigenvalues


ModuleNotFoundError: No module named 'lot3_algorithmes'

## temp d'executtion

In [ ]:

def calculer_metriques(X, V, V_direct):
    """
    Calcule les erreurs de reconstruction,
    d'orthogonalite et de projecteur.
    """

    q = V.shape[1]

    # Reconstruction
    X_reconstructed = X @ V @ V.T

    erreur_reconstruction = (
        np.linalg.norm(X - X_reconstructed, ord="fro")**2
        /
        np.linalg.norm(X, ord="fro")**2
    )

    # Orthogonalite
    I = np.eye(q)

    erreur_orthogonalite = np.linalg.norm(
        V.T @ V - I,
        ord="fro"
    )

    # Difference entre les projecteurs
    P_direct = V_direct @ V_direct.T
    P_iterative = V @ V.T

    difference_projecteurs = np.linalg.norm(
        P_direct - P_iterative,
        ord="fro"
    )

    return {
        "erreur_reconstruction": erreur_reconstruction,
        "erreur_orthogonalite": erreur_orthogonalite,
        "difference_projecteurs": difference_projecteurs
    }


In [ ]:

def experimenter_comparaison(
    type_spectre,
    q=10,
    iterations=[5, 10, 20, 50]
):
    """
    Compare ACP directe et ACP iterative
    pour un spectre donne.
    """

    # 1. Generer les donnees
    X, valeurs_theoriques = generer_donnees_synthetiques(
        n=500,
        d=64,
        type_spectre=type_spectre
    )

    # 2. Centrer X
    X = X - np.mean(X, axis=0)

    # 3. Eigengap theorique
    eigengap = (
        valeurs_theoriques[q - 1]
        - valeurs_theoriques[q]
    )

    # 4. ACP directe
    debut = time.perf_counter()

    model_direct = acp_directe_fit(X, q)

    temps_direct = time.perf_counter() - debut

    V_direct = model_direct["components"]

    # 5. Afficher les resultats
    resultats = []

    for nb_iter in iterations:

        debut = time.perf_counter()

        model_iterative = acp_iterative_fit(
            X,
            q,
            max_iter=nb_iter,
            tol=0,
            seed=42
        )

        temps_iterative = time.perf_counter() - debut

        V_iterative = model_iterative["components"]

        # Calcul des metriques
        metriques = calculer_metriques(
            X,
            V_iterative,
            V_direct
        )

        resultats.append({
            "iterations": nb_iter,
            "temps_direct": temps_direct,
            "temps_iterative": temps_iterative,
            "eigengap": eigengap,
            "converged": model_iterative["converged"],
            **metriques
        })

    return resultats
